# SO(3) equivariant-polynomial examples

A minimal example of the high-level `equivariant_polynomials` API.  We compute
minimal integrity bases for an SO(3) example and time the construction:

* input representation `X = 2 H_2 (+) H_3` (two copies of the `l=2` irrep and one `l=3`),
* output representation `Y = H_4` (the `l=4` irrep),
* degree bound `D_max = 9`.

`invariant_generators(X)` returns minimal algebra generators of the invariant
algebra `R_G(X)`; `covariant_generators(X, Y)` returns minimal module generators
of the covariant module `M_G(X, Y)`.

In [1]:
import sys
from pathlib import Path

# Make the in-repo package importable without installing it.
root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "equivariant_polynomials").is_dir()
)
sys.path.insert(0, str(root / "src"))
print(f"Project root: {root}")

Project root: c:\Users\micha\OneDrive\Desktop\Oden\Notes\Equivariant Polynomial Tensor Functions\Python\equivariant_polynomials


In [2]:
from time import perf_counter

from equivariant_polynomials import (
    SO3RepresentationTheory,
    Representation,
    Options,
    invariant_generators,
    covariant_generators,
    invariant_hilbert_series,
    covariant_hilbert_series,
)

theory = SO3RepresentationTheory()
domain = Representation(theory, [(2, 2), (3, 1)], name="X")
codomain = Representation.irrep(theory, 4, name="Y")
max_degree = 9
options = Options(modulus=2521, seed=12345)

domain, codomain

(Representation(2*H[2] + H[3], dim=17 'X'), Representation(H[4], dim=9 'Y'))

## Benchmark

In [3]:
start = perf_counter()
invariants = invariant_generators(domain, max_degree=max_degree, options=options)
algebra_seconds = perf_counter() - start
print(f"algebra generators: {algebra_seconds:.3f} s")

start = perf_counter()
covariants = covariant_generators(domain, codomain, max_degree=max_degree, options=options)
module_seconds = perf_counter() - start
print(f"module generators:  {module_seconds:.3f} s")
print(f"total:              {algebra_seconds + module_seconds:.3f} s")

algebra generators: 2.145 s
module generators:  5.208 s
total:              7.353 s


In [4]:
print("invariant algebra")
print("  Hilbert series   :", invariant_hilbert_series(domain, max_degree))
print("  generator counts :", invariants.counts_by_degree())
print()
print("covariant module M(X, H_4)")
print("  Hilbert series   :", covariant_hilbert_series(domain, codomain, max_degree))
print("  generator counts :", covariants.counts_by_degree())

invariant algebra
  Hilbert series   : (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739)
  generator counts : (1, 0, 4, 7, 14, 26, 52, 68, 60, 39)

covariant module M(X, H_4)
  Hilbert series   : (0, 0, 6, 21, 87, 273, 807, 2109, 5205, 11919)
  generator counts : (0, 0, 6, 21, 63, 147, 264, 284, 133, 35)


## Evaluating the generators

The returned objects are callable.  A point of `X` is given in isotypic
coordinates: one array per component, of shape `(multiplicity, dim H_irrep)`.
Invariants evaluate to scalars (`H_0` is one-dimensional); the covariants here
evaluate into `H_4` (dimension 9).

In [5]:
import numpy as np

rng = np.random.default_rng(0)
x = (
    rng.standard_normal((2, 5)),  # 2 copies of H_2 (dim 5)
    rng.standard_normal((1, 7)),  # 1 copy  of H_3 (dim 7)
)

quadratic_invariants = [g(x) for g in invariants if g.degree == 2]
print("degree-2 invariant values:", np.round(np.concatenate(quadratic_invariants).real, 4))
print("a covariant value lives in H_4, shape", covariants[0](x).shape)

degree-2 invariant values: [ 1.11184015e+06 -6.84448000e+01 -3.56138000e+02 -3.53446500e+02]
a covariant value lives in H_4, shape (9,)
